# Reworld Haverhill — Dashboard Launcher

Run **Cell 1** to start the server. The URL is printed below the cell.

| Button in dashboard | What it does |
|---|---|
| ⟳ **Refresh** | Re-reads CSVs from disk and redraws all charts |
| ⏱ **Auto-refresh: OFF** | Toggle — auto-refreshes every 5 min when ON |
| ☁ **Sync from S3** | Pulls latest camera data from S3, recomputes signals (~30–60 s), then auto-refreshes |

Run **Cell 3** to stop the server. The server stays alive as long as this kernel is alive.

> **No port-opening needed.** `jupyter-server-proxy` tunnels everything through JupyterHub's existing HTTPS connection automatically.

In [15]:
import os, sys, time, subprocess, pathlib, socket
import psutil
from IPython.display import display, HTML

PORT     = 8050

LOG_FILE = '/tmp/dashboard_uvicorn.log'
PYTHON   = sys.executable

# ── 1. kill any previous instance (pure Python — no lsof/fuser needed) ───────
def kill_port(port):
    killed = []
    for p in psutil.process_iter(['pid']):
        try:
            for conn in p.net_connections():
                if conn.laddr.port == port:
                    p.kill()
                    killed.append(p.pid)
                    break
        except (psutil.NoSuchProcess, psutil.AccessDenied):
            pass
    if killed:
        time.sleep(0.6)
    return killed

killed = kill_port(PORT)
if killed:
    print(f'Stopped previous server (pid {killed})')

# ── 2. start uvicorn as a subprocess so startup errors are visible ────────────
log  = open(LOG_FILE, 'w')
proc = subprocess.Popen(
    [PYTHON, '-m', 'uvicorn', 'dashboard_api:app',
     '--host', '0.0.0.0', '--port', str(PORT), '--log-level', 'warning', '--app-dir', '/home/shared/kumar/library/fastapi-jupyter-dashboard'],
    cwd='/home/shared/kumar/library/fastapi-jupyter-dashboard',
    stdout=log, stderr=log,
)

# ── 3. wait until the port accepts connections ────────────────────────────────
for _ in range(25):
    try:
        s = socket.create_connection(('localhost', PORT), timeout=0.3)
        s.close()
        break
    except OSError:
        time.sleep(0.3)
else:
    log.flush()
    err = pathlib.Path(LOG_FILE).read_text()
    display(HTML(
        '<pre style="color:#ef4444;background:#1a1f2e;padding:12px;border-radius:8px">'
        f'Server failed to start. Log:\n\n{err or "(empty)"}'
        '</pre>'
    ))
    raise RuntimeError('Dashboard server did not start — see error above')

# ── 4. print the shareable URL ────────────────────────────────────────────────
base = os.environ.get('JUPYTERHUB_PUBLIC_URL', f'http://localhost:{PORT}/')
url  = base.rstrip('/') + f'/proxy/{PORT}/'

display(HTML(f'''
<div style="background:#1a1f2e;border:1px solid rgba(34,197,94,.35);border-radius:10px;
            padding:16px 20px;font-family:Inter,sans-serif;max-width:720px">
  <div style="font-size:13px;color:#86efac;font-weight:700;margin-bottom:8px">
    ✓ Dashboard running on port {PORT}
  </div>
  <div style="font-size:13px;color:#94a3b8;margin-bottom:10px">
    Share this link with demo attendees (JupyterHub login required):
  </div>
  <a href="{url}" target="_blank"
     style="background:#0f1117;border:1px solid rgba(59,130,246,.4);border-radius:7px;
            padding:9px 14px;font-size:13px;color:#93c5fd;text-decoration:none;
            display:inline-block;word-break:break-all">{url}</a>
  <div style="font-size:11px;color:#475569;margin-top:12px;line-height:1.6">
    No port-opening needed — routed through JupyterHub HTTPS automatically.<br/>
    If the page shows errors, run the <strong>Check log</strong> cell below.
  </div>
</div>
'''))

Stopped previous server (pid [5845])


In [7]:
# ── Check server log (run this if the dashboard shows errors) ─────────────────
import pathlib
txt = pathlib.Path('/tmp/dashboard_uvicorn.log').read_text()
print(txt if txt.strip() else '(log is empty — server is healthy)')

(log is empty — server is healthy)


In [10]:
# ── Stop the server ───────────────────────────────────────────────────────────
try:
    proc.terminate()
    proc.wait(timeout=3)
    print(f'Server stopped (exit code {proc.returncode})')
except Exception as e:
    print('Stop error:', e)

Server stopped (exit code -15)
